# 09 — MCP Integration & Framework Adapters

This notebook covers integrating Hindsight with agent frameworks:
- MCP Server: the universal integration
- Direct HTTP API (building custom integrations)
- LangChain integration pattern
- LiteLLM callback pattern
- Building a memory-aware agent from scratch

In [ ]:
from hindsight_client import Hindsight
import httpx
import json
import asyncio

HINDSIGHT_URL = "http://localhost:8888"
BANK_ID = "tutorial-mcp"

client = Hindsight(base_url=HINDSIGHT_URL)

# Create a bank for integration testing
client.create_bank(
    bank_id=BANK_ID,
    name="MCP Integration Tutorial",
    reflect_mission="""
    I am an agent with persistent memory.
    I remember user preferences, past conversations, and project context.
    """
)
print(f"✓ Bank '{BANK_ID}' ready")

## 1. Direct HTTP API — Building Custom Integrations

For frameworks without an official integration, use the HTTP API directly.

In [ ]:
class HindsightHTTPClient:
    """Minimal HTTP client for Hindsight — no SDK dependency needed.
    
    Use this when you need to integrate Hindsight into a language
    or framework that doesn't have an official SDK.
    """
    
    def __init__(self, base_url, api_key=None):
        self.base_url = base_url.rstrip('/')
        self.headers = {"Content-Type": "application/json"}
        if api_key:
            self.headers["Authorization"] = f"Bearer {api_key}"
        self.client = httpx.AsyncClient(timeout=30.0, headers=self.headers)
    
    async def retain(self, bank_id, content, **kwargs):
        """Store a memory."""
        item = {"content": content, **kwargs}
        resp = await self.client.post(
            f"{self.base_url}/v1/default/banks/{bank_id}/memories",
            json={"items": [item]}
        )
        resp.raise_for_status()
        return resp.json()
    
    async def recall(self, bank_id, query, **kwargs):
        """Search memories."""
        body = {"query": query, **kwargs}
        resp = await self.client.post(
            f"{self.base_url}/v1/default/banks/{bank_id}/memories/recall",
            json=body
        )
        resp.raise_for_status()
        return resp.json()
    
    async def reflect(self, bank_id, query, **kwargs):
        """Reason over memories."""
        body = {"query": query, **kwargs}
        resp = await self.client.post(
            f"{self.base_url}/v1/default/banks/{bank_id}/reflect",
            json=body
        )
        resp.raise_for_status()
        return resp.json()
    
    async def close(self):
        await self.client.aclose()

print("✓ HindsightHTTPClient defined")
print("  Use this pattern for any language: Go, Rust, Node.js, etc.")

In [ ]:
# Test the HTTP client
async def test_http_client():
    api = HindsightHTTPClient(HINDSIGHT_URL)
    
    # Retain via HTTP
    result = await api.retain(
        BANK_ID,
        "HTTP test: The user prefers dark mode UIs"
    )
    print(f"✓ Retain: {result.get('success', '?')}")
    
    # Recall via HTTP
    results = await api.recall(
        BANK_ID,
        "UI preferences",
        types=["world"]
    )
    print(f"✓ Recall: {len(results.get('results', []))} results")
    
    # Reflect via HTTP
    answer = await api.reflect(
        BANK_ID,
        "What UI preferences does the user have?"
    )
    print(f"✓ Reflect: {answer.get('text', '')[:100]}...")
    
    await api.close()

await test_http_client()

## 2. MCP Server Integration Pattern

The MCP (Model Context Protocol) server exposes Hindsight as tools.
This is the universal integration — any MCP client can use it.

### Starting the MCP Server

```bash
# Standalone MCP server
hindsight-local-mcp

# Or via Docker
docker run -p 8888:8888 ghcr.io/vectorize-io/hindsight:latest
# MCP endpoint available at http://localhost:8888/mcp
```

### MCP Tools Available

| Tool | Description | Parameters |
|------|-------------|------------|
| `hindsight_retain` | Store a memory | `bank_id`, `content`, `context`, `timestamp`, `metadata` |
| `hindsight_recall` | Search memories | `bank_id`, `query`, `types`, `budget`, `max_tokens` |
| `hindsight_reflect` | Reason over memories | `bank_id`, `query`, `budget`, `context` |
| `hindsight_create_mental_model` | Create curated knowledge | `bank_id`, `name`, `content`, `tags` |
| `hindsight_list_mental_models` | List mental models | `bank_id` |

## 3. Building a Memory-Aware Agent from Scratch

Here's a complete pattern for an agent that automatically recalls context
before each turn and retains the conversation after.

In [ ]:
class MemoryAwareAgent:
    """An agent that uses Hindsight for persistent memory.
    
    Pattern:
    1. On each user message: recall relevant past context
    2. Generate response (you'd plug in your LLM here)
    3. Retain the exchange for future recall
    """
    
    def __init__(self, base_url, bank_id, agent_name="Assistant"):
        self.client = Hindsight(base_url=base_url)
        self.bank_id = bank_id
        self.agent_name = agent_name
        self.session_id = f"session-{int(asyncio.get_event_loop().time())}"
        self.turn = 0
    
    def get_context(self, user_message):
        """Recall relevant memories to inject as context."""
        results = self.client.recall(
            bank_id=self.bank_id,
            query=user_message,
            budget="mid",
            max_tokens=2048
        )
        
        if not results.results:
            return "(No relevant past context.)"
        
        lines = ["Relevant context from past interactions:"]
        for r in results.results:
            lines.append(f"- [{r.type}] {r.text}")
        return "\n".join(lines)
    
    def handle_message(self, user_message):
        """Process a user message with memory augmentation."""
        self.turn += 1
        
        # Step 1: Recall relevant context
        context = self.get_context(user_message)
        
        # Step 2: Build augmented prompt (you'd call your LLM here)
        augmented_prompt = f"""{context}

User message: {user_message}

Using the context above, respond to the user.
Remember: you have persistent memory — your response will be remembered."""
        
        # Step 3: In a real agent, call LLM here
        # response = llm.generate(augmented_prompt)
        # For this demo, we'll use reflect() itself as the "LLM"
        answer = self.client.reflect(
            bank_id=self.bank_id,
            query=augmented_prompt,
            max_tokens=512
        )
        agent_response = answer.text
        
        # Step 4: Retain the exchange
        self.client.retain(
            bank_id=self.bank_id,
            content=f"User: {user_message}",
            context="user-message",
            document_id=self.session_id,
            metadata={"turn": str(self.turn), "role": "user"}
        )
        self.client.retain(
            bank_id=self.bank_id,
            content=f"{self.agent_name}: {agent_response}",
            context="agent-response",
            document_id=self.session_id,
            metadata={"turn": str(self.turn), "role": "agent"}
        )
        
        return agent_response

# Create and test the agent
agent = MemoryAwareAgent(HINDSIGHT_URL, BANK_ID, "MemoryBot")
print("✓ MemoryAwareAgent created")
print(f"  Session: {agent.session_id}")
print(f"  Bank: {agent.bank_id}")
print()

# First interaction
response1 = agent.handle_message("Hi! My name is Alex and I'm building a weather app with React.")
print(f"User: Hi! My name is Alex and I'm building a weather app with React.")
print(f"Bot: {response1[:200]}...")
print()

# Second interaction — should recall Alex's name and project
response2 = agent.handle_message("What framework did I say I'm using for my app?")
print(f"User: What framework did I say I'm using for my app?")
print(f"Bot: {response2[:200]}...")
print()

# Third interaction — builds on context
response3 = agent.handle_message("Can you suggest a state management library for it?")
print(f"User: Can you suggest a state management library for it?")
print(f"Bot: {response3[:200]}...")

## 4. LangChain-Style Integration Pattern

If you're using LangChain or a similar framework, here's how to create
a Hindsight memory tool.

In [ ]:
class HindsightMemoryTool:
    """A tool that agents can call to store and retrieve memories.
    
    This follows the LangChain BaseTool pattern — the agent decides
    when to use retain vs recall based on the situation.
    """
    
    def __init__(self, base_url, bank_id):
        self.client = Hindsight(base_url=base_url)
        self.bank_id = bank_id
        self.name = "hindsight_memory"
        self.description = """
        Use this tool to manage persistent memory:
        - To STORE information: use action='retain' with content
        - To SEARCH memories: use action='recall' with a query
        - To REASON about memories: use action='reflect' with a query
        """
    
    def run(self, action, content=None, query=None):
        """Execute a memory operation.
        
        Args:
            action: 'retain', 'recall', or 'reflect'
            content: The text to store (for retain)
            query: The search/reasoning query (for recall/reflect)
        """
        if action == "retain" and content:
            result = self.client.retain(
                bank_id=self.bank_id,
                content=content
            )
            return f"Memory stored: {content[:100]}..."
        
        elif action == "recall" and query:
            results = self.client.recall(
                bank_id=self.bank_id,
                query=query,
                budget="mid"
            )
            if not results.results:
                return "No relevant memories found."
            return "\n".join([f"- {r.text}" for r in results.results])
        
        elif action == "reflect" and query:
            answer = self.client.reflect(
                bank_id=self.bank_id,
                query=query
            )
            return answer.text
        
        else:
            return "Error: Must specify action ('retain', 'recall', 'reflect') and content/query."

# Test the tool
tool = HindsightMemoryTool(HINDSIGHT_URL, BANK_ID)

print("=== Testing HindsightMemoryTool ===\n")

# Store
print("1. Retain:")
print(tool.run(action="retain", content="The production database is PostgreSQL 17 with pgvector."))

print("\n2. Recall:")
print(tool.run(action="recall", query="database"))

print("\n3. Reflect:")
print(tool.run(action="reflect", query="What database do we use?")[:300])

## 5. LiteLLM Callback Pattern (Zero Code Change)

For apps using LiteLLM, Hindsight can be added with zero code changes.

```yaml
# litellm_config.yaml
litellm_settings:
  callbacks: ["hindsight_litellm"]
  hindsight:
    bank_id: "litellm-agent"
    api_url: "http://localhost:8888"
    auto_retain: true        # Store every call
    auto_recall: true        # Inject memories into context
```

After this config, every LLM call through LiteLLM will:
1. Recall relevant memories and inject them into the prompt
2. Retain the user message and assistant response

No Python code changes needed.

## 6. Claude Code / Cursor Integration Setup

To add Hindsight memory to Claude Code or Cursor:

### Claude Code

Add to `~/.claude/claude_desktop_config.json` or project `.mcp.json`:

```json
{
  "mcpServers": {
    "hindsight": {
      "command": "hindsight-local-mcp",
      "env": {
        "HINDSIGHT_API_LLM_PROVIDER": "openai",
        "HINDSIGHT_API_LLM_API_KEY": "***"
      }
    }
  }
}
```

Or connect to a running server:

```json
{
  "mcpServers": {
    "hindsight": {
      "type": "http",
      "url": "http://localhost:8888/mcp"
    }
  }
}
```

### Cursor

1. Install the Hindsight extension or configure MCP in Cursor settings
2. Cursor will automatically recall context at session start
3. Conversations are retained for future sessions

## 7. Integration Decision Matrix

Use this flowchart to choose the right integration approach:

In [ ]:
integration_guide = """
┌──────────────────────────────────────────────────┐
│           How to Integrate Hindsight?              │
├──────────────────────────────────────────────────┤
│                                                    │
│  Using Claude Code / Cursor / Continue?            │
│    → MCP Server (hindsight-local-mcp)               │
│                                                    │
│  Using LangChain / LangGraph?                      │
│    → pip install hindsight-langchain               │
│    → HindsightToolSpec + HindsightMemoryStore       │
│                                                    │
│  Using CrewAI?                                     │
│    → pip install hindsight-crewai                  │
│    → Agent(memory=HindsightMemory(...))             │
│                                                    │
│  Using Pydantic AI?                                │
│    → pip install hindsight-pydantic-ai             │
│    → Agent(memory_provider=HindsightMemoryProvider) │
│                                                    │
│  Using OpenAI Agents SDK?                          │
│    → pip install hindsight-openai-agents           │
│                                                    │
│  Using LiteLLM proxy?                              │
│    → hindsight-litellm (ZERO code changes!)         │
│                                                    │
│  Using anything with MCP support?                  │
│    → MCP Server (universal)                         │
│                                                    │
│  Custom / no framework?                            │
│    → hindsight-client (Python SDK)                  │
│    → Direct HTTP API (any language)                 │
│                                                    │
└──────────────────────────────────────────────────┘
"""
print(integration_guide)

## 8. Cleanup

In [ ]:
# client.delete_bank(bank_id=BANK_ID)
print("Bank preserved.")

## Summary

You learned:
1. **Direct HTTP API** — building custom integrations without the SDK
2. **MCP Server** — the universal integration pattern
3. **Memory-Aware Agent** — complete pattern: recall → respond → retain
4. **LangChain-style tool** — agent-decides-when-to-use pattern
5. **LiteLLM callback** — zero code change integration
6. **Claude Code / Cursor setup** — MCP configuration
7. **Decision matrix** — choosing the right integration approach

**You've completed the Hindsight tutorial series!**

See [appendix.md](../appendix.md) for the full API reference, cookbook links, and glossary.